# NB09 — Amazon Fine Food Reviews: COA, AdaBoost, ablation, Wilcoxon (auxiliary experiment)

> **Changed on 2026-10-12** (original with outputs: `Code/Newest/Archive/pre-es-fix_2026-10-12/09-amazon-fine-food-coa-adaboost.ipynb`).
> Results produced by the old version must not be used. Background: `Decision/004-early-stopping-zero-label-bug.md`,
> `Decision/006-rigor-upgrades.md`.

**What changed and why**

| Part | Old behaviour | New behaviour | Why |
|---|---|---|---|
| Early stopping everywhere (COA-ANN and COA-CNN fitness, final training, ablation, Wilcoxon) | `validation_data` labels were `np.zeros(...)` | true labels | zero labels made early stopping restore the epoch closest to "predict all negative" |
| Ablation "Full" rows | copied from the final-training run (≤100/30 epochs) | retrained with the same schedule as the variants (≤50/20 epochs) | otherwise Full-vs-variant differences mix two effects |
| Wilcoxon seeds | 5 | 10 | with 5 paired seeds the exact two-sided test cannot reach p < 0.0625 |

Unchanged: data subset (20k negative / 40k positive, seed 42), preprocessing (MAX_VOCAB 20,000, MAX_LEN rule), 60/20/20 split,
COA settings (8×8 ANN, 6×4 CNN on a 15,000-sample fitness subset), AdaBoost settings, plots.

This notebook is self-contained (reads the Kaggle dataset `snap/amazon-fine-food-reviews`). **Accelerator:** GPU T4 x2.

# Amazon Fine Food Reviews — COA Pipeline + AdaBoost (Single Notebook)
Dataset: `snap/amazon-fine-food-reviews` (Reviews.csv)

**Small subset** to stay well within Kaggle GPU time limits.

**What this does:**
1. Load + filter 3-star reviews, subsample to **60k** (20k neg + 40k pos)
2. Preprocess, FastText embeddings, 60/20/20 split
3. COA-ANN search (8 coyotes × 8 iters, single GPU)
4. COA-CNN search (6 coyotes × 4 iters, single GPU, 15k fitness subset)
5. Final training: TF-IDF×2, Default/COA ANN, Default/COA CNN+BiLSTM (MirroredStrategy)
6. **AdaBoost** on FastText embeddings (depth-1 stumps + depth-3 trees)
7. Ablation study
8. Wilcoxon test (5 seeds, ANN + CNN only — AdaBoost excluded due to its own
   determinism across seeds being irrelevant to gradient-based training)
9. All comparison figures

**No K-Fold CV** (omitted to stay within GPU time limit)

**Kaggle dataset to add:** `snap/amazon-fine-food-reviews`
**Estimated total runtime:** ~2-2.5 hours


In [ ]:
import os, random as _random, warnings, re, string, json, gc, pickle, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import gensim.downloader as gensim_api
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
    matthews_corrcoef, average_precision_score, classification_report,
    precision_score, recall_score, confusion_matrix,
    precision_recall_curve, roc_curve)
from scipy.stats import wilcoxon
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords",  quiet=True)

# ── Constants ─────────────────────────────────────────────────────────────────
SEED       = 42
MAX_VOCAB  = 20000
EMBED_DIM  = 300
DATASET    = "amazon_fine_food"

# ── Subsample sizes — SMALL, keeps total runtime ~2-2.5 hours ────────────────
N_NEG      = 20000   # negative samples (1-2 star)
N_POS      = 40000   # positive samples (4-5 star) — more available naturally
# Total: 60k -> 60/20/20 split -> 36k train / 12k val / 12k test

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH  = "/kaggle/input/datasets/organizations/snap/amazon-fine-food-reviews/Reviews.csv"
WORKING    = "/kaggle/working"
SHARED_DIR = f"{WORKING}/shared_data/{DATASET}"
RESULTS_DIR= f"{WORKING}/saved_results/{DATASET}"
MODELS_DIR = f"{WORKING}/saved_models/{DATASET}"
for d in [SHARED_DIR, RESULTS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/global", exist_ok=True)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed); np.random.seed(seed)

set_seeds()
print(f"SEED={SEED}  MAX_VOCAB={MAX_VOCAB}  N_NEG={N_NEG:,}  N_POS={N_POS:,}")
print(f"DATA_PATH: {DATA_PATH}")
print(f"Total subset: {N_NEG+N_POS:,} samples")


## 1. Load Dataset

In [ ]:
print("Loading Reviews.csv...")
df = pd.read_csv(DATA_PATH)
print(f"Raw rows: {len(df):,}")

# Remove 3-star (neutral) — CRITICAL: do not map 3-star to negative
df = df[df["Score"] != 3].reset_index(drop=True)
print(f"After removing 3-star: {len(df):,}")

# Binary labels: 4,5 -> positive (1), 1,2 -> negative (0)
df["label"] = (df["Score"] >= 4).astype(int)

neg_df = df[df["label"]==0]
pos_df = df[df["label"]==1]
print(f"Available — Negative (1-2★): {len(neg_df):,}  Positive (4-5★): {len(pos_df):,}")

# Subsample to small fixed sizes
neg_sample = neg_df.sample(n=min(N_NEG, len(neg_df)), random_state=SEED)
pos_sample = pos_df.sample(n=min(N_POS, len(pos_df)), random_state=SEED)
df_sub = pd.concat([neg_sample, pos_sample]).sample(frac=1, random_state=SEED).reset_index(drop=True)

texts_raw = df_sub["Text"].fillna("").tolist()
labels    = df_sub["label"].values
print(f"\nSubset used: {len(df_sub):,} | pos={labels.sum():,} neg={(labels==0).sum():,}")
print(f"Imbalance ratio: {labels.sum()/(labels==0).sum():.2f}:1")


## 2. Preprocess

In [ ]:
stop_words = set(stopwords.words("english"))

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(
        "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]+",
        "", text, flags=re.UNICODE)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = word_tokenize(text)
    return [t for t in tokens if t not in stop_words and len(t) > 2]

print(f"Preprocessing {len(texts_raw):,} texts...")
tokens_list  = [preprocess(t) for t in texts_raw]
strings_list = [" ".join(tok) for tok in tokens_list]
print(f"Done. Example: {tokens_list[0][:8]}")

# MAX_LEN: 92nd percentile of lengths, rounded to multiple of 8
lengths = [len(tok) for tok in tokens_list]
pct92   = int(np.percentile(lengths, 92))
MAX_LEN = max(int(np.ceil(pct92/8)*8), 50)
print(f"\nLength stats: mean={np.mean(lengths):.0f}  median={np.median(lengths):.0f}  "
      f"92nd pct={pct92}")
print(f"MAX_LEN={MAX_LEN} (covers "
      f"{100*sum(1 for l in lengths if l<=MAX_LEN)/len(lengths):.1f}% of reviews)")


## 3. FastText Embeddings

In [ ]:
print("Loading FastText (fasttext-wiki-news-subwords-300)...")
ft = gensim_api.load("fasttext-wiki-news-subwords-300")
print(f"FastText: dim={ft.vector_size}  vocab={len(ft.key_to_index):,}")

def mean_embed(tokens):
    vecs = [ft[t] for t in tokens if t in ft]
    return (np.mean(vecs, axis=0).astype(np.float32)
            if vecs else np.zeros(EMBED_DIM, dtype=np.float32))

print(f"Computing sentence embeddings ({len(tokens_list):,})...")
X_emb = np.array([mean_embed(tok) for tok in tokens_list], dtype=np.float32)
print(f"X_emb: {X_emb.shape}")

print("Tokenizing sequences...")
tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(strings_list)
seqs  = tokenizer.texts_to_sequences(strings_list)
X_seq = pad_sequences(seqs, maxlen=MAX_LEN,
                       padding="post", truncating="post").astype(np.int32)
print(f"X_seq: {X_seq.shape}")

vocab_size = min(MAX_VOCAB, len(tokenizer.word_index)+1)
emb_mat    = np.zeros((vocab_size, EMBED_DIM), dtype=np.float32)
found = 0
for word, idx in tokenizer.word_index.items():
    if idx < vocab_size and word in ft:
        emb_mat[idx] = ft[word]; found += 1
cov_pct = round(100*found/vocab_size, 2)
print(f"Embedding matrix: {emb_mat.shape}  Coverage: {cov_pct}%")

del ft; gc.collect()
print("FastText freed.")


## 4. 60/20/20 Stratified Split

In [ ]:
tv_idx, test_idx = train_test_split(
    np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=SEED)
train_idx, val_idx = train_test_split(
    tv_idx, test_size=0.25, stratify=labels[tv_idx], random_state=SEED)

X_emb_train=X_emb[train_idx]; X_emb_val=X_emb[val_idx]; X_emb_test=X_emb[test_idx]
X_seq_train=X_seq[train_idx]; X_seq_val=X_seq[val_idx]; X_seq_test=X_seq[test_idx]
y_train=labels[train_idx];    y_val=labels[val_idx];     y_test=labels[test_idx]
X_str_train=[strings_list[i] for i in train_idx]
X_str_val  =[strings_list[i] for i in val_idx]
X_str_test =[strings_list[i] for i in test_idx]
test_texts_raw=[texts_raw[i] for i in test_idx]

cw = dict(zip(
    [int(k) for k in np.unique(y_train)],
    [float(v) for v in compute_class_weight(
        "balanced", classes=np.unique(y_train), y=y_train)]))

print(f"Train : {len(y_train):,}  Val: {len(y_val):,}  Test: {len(y_test):,}")
print(f"class_weights: {cw}")

np.save(f"{SHARED_DIR}/X_emb_train.npy", X_emb_train)
np.save(f"{SHARED_DIR}/X_emb_val.npy",   X_emb_val)
np.save(f"{SHARED_DIR}/X_emb_test.npy",  X_emb_test)
np.save(f"{SHARED_DIR}/X_seq_train.npy", X_seq_train)
np.save(f"{SHARED_DIR}/X_seq_val.npy",   X_seq_val)
np.save(f"{SHARED_DIR}/X_seq_test.npy",  X_seq_test)
np.save(f"{SHARED_DIR}/y_train.npy",     y_train)
np.save(f"{SHARED_DIR}/y_val.npy",       y_val)
np.save(f"{SHARED_DIR}/y_test.npy",      y_test)
np.save(f"{SHARED_DIR}/emb_matrix.npy",  emb_mat)
with open(f"{SHARED_DIR}/X_txt_test.txt","w",encoding="utf-8") as f:
    f.write("\n".join(X_str_test))
with open(f"{SHARED_DIR}/test_texts_raw.txt","w",encoding="utf-8") as f:
    f.write("\n".join(test_texts_raw))
with open(f"{SHARED_DIR}/tokenizer.json","w") as f: f.write(tokenizer.to_json())
json.dump(cw, open(f"{SHARED_DIR}/class_weights.json","w"), indent=2)
json.dump({"dataset":DATASET,"max_len":MAX_LEN,"max_vocab":MAX_VOCAB,
           "vocab_size":vocab_size,"embed_dim":EMBED_DIM,"seed":SEED,
           "emb_coverage_pct":cov_pct,
           "n_train":int(len(y_train)),"n_val":int(len(y_val)),"n_test":int(len(y_test)),
           "n_pos_train":int(y_train.sum()),"n_neg_train":int((y_train==0).sum()),
           "imbalance_ratio":round(float(y_train.sum()/max((y_train==0).sum(),1)),4)},
          open(f"{SHARED_DIR}/config.json","w"), indent=2)
print("Saved all arrays.")


## 5. GPU Setup

In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# IMPORTANT: we do NOT call tf.config.set_visible_devices() here.
# That call is irreversible for the lifetime of the process — if used to pin
# COA to GPU 0 only, it would also silently restrict the later MirroredStrategy
# (Cell 17, final training) to a single GPU, halving throughput with no error.
# Instead, COA explicitly runs its ops inside `with tf.device("/GPU:0"):` further
# below, which is scoped per-call and does not affect global device visibility.
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError as e:
        # Already initialized earlier in this kernel session (e.g. cell re-run) — safe to ignore.
        print(f"Note: memory growth already set for {g.name} ({e}).")

COA_DEVICE = "/GPU:0" if gpus else "/CPU:0"
print(f"GPUs visible: {len(gpus)}  |  COA will run on: {COA_DEVICE}  "
      f"|  Final training (Cell 17+) will use all {len(gpus)} GPU(s) via MirroredStrategy")

tf.keras.mixed_precision.set_global_policy("mixed_float16")

ANN_BATCH_COA = 512
CNN_BATCH_COA = 64

def set_seeds_tf(seed=SEED):
    set_seeds(seed); tf.random.set_seed(seed)

def make_es(patience=5, verbose=0):
    return EarlyStopping(monitor="val_loss", patience=patience,
                         restore_best_weights=True, verbose=verbose)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype(np.float32), y.astype(np.float32)))
    if shuffle: ds = ds.shuffle(min(len(y),50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def make_seq_ds(X, y, batch, shuffle=True, seed=SEED):
    ds = tf.data.Dataset.from_tensor_slices(
        (X.astype(np.int32), y.astype(np.float32)))
    if shuffle: ds = ds.shuffle(min(len(y),50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true,y_prob),4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true,y_prob),4) if y_prob is not None else float("nan")
    rep = classification_report(y_true,y_pred,output_dict=True,zero_division=0)
    return {"Model":name,
            "Accuracy"   :round(accuracy_score(y_true,y_pred),4),
            "Precision"  :round(precision_score(y_true,y_pred,zero_division=0),4),
            "Recall"     :round(recall_score(y_true,y_pred,zero_division=0),4),
            "F1-score"   :round(f1_score(y_true,y_pred,zero_division=0),4),
            "Macro-F1"   :round(rep["macro avg"]["f1-score"],4),
            "Weighted-F1":round(rep["weighted avg"]["f1-score"],4),
            "MCC"        :round(matthews_corrcoef(y_true,y_pred),4),
            "AUC-ROC":auc,"Avg-Precision":ap}

def best_threshold(y_true, y_prob):
    prec,rec,thr = precision_recall_curve(y_true,y_prob)
    f1s = 2*prec*rec/(prec+rec+1e-9)
    return float(thr[np.argmax(f1s[:-1])])

set_seeds_tf()
print("GPU setup complete.")


## 6. COA — ANN Hyperparameter Search\n\nFitness is evaluated **exclusively on the validation set** — `y_test` is never referenced inside `ann_fitness`.

In [ ]:
ANN_BOUNDS = np.array([[64,256],[32,128],[-4,-2]], dtype=float)

def decode_ann(pos):
    return (int(round(np.clip(pos[0],64,256))),
            int(round(np.clip(pos[1],32,128))),
            float(10**np.clip(pos[2],-4,-2)))

def rand_coyote(bounds, rng):
    return np.array([rng.uniform(lo,hi) for lo,hi in bounds])

def clip_pos(pos, bounds):
    return np.clip(pos, bounds[:,0], bounds[:,1])

# Pre-build val dataset for COA (reused across all evaluations) — uses y_val ONLY
val_ds_ann = make_tf_ds(X_emb_val, np.zeros(len(y_val)),
                         ANN_BATCH_COA*4, shuffle=False)
n_int      = int(len(y_train)*0.1)
X_fit_tr   = X_emb_train[:-n_int]; y_fit_tr = y_train[:-n_int]
X_fit_iv   = X_emb_train[-n_int:]; y_fit_iv = y_train[-n_int:]
eval_cnt   = [0]

def ann_fitness(pos):
    """Trains on X_fit_tr/y_fit_tr, evaluates F1 on X_emb_val/y_val.
    y_test is NEVER referenced in this function. Runs on COA_DEVICE (single GPU)
    so it never restricts global GPU visibility for later MirroredStrategy use."""
    with tf.device(COA_DEVICE):
        eval_cnt[0] += 1
        h1,h2,lr = decode_ann(pos)
        set_seeds_tf(SEED + eval_cnt[0])
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(h1,activation="relu",input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2,activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32"),
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss="binary_crossentropy",metrics=["accuracy"])
        model.fit(make_tf_ds(X_fit_tr,y_fit_tr,ANN_BATCH_COA),
                  validation_data=make_tf_ds(X_fit_iv,y_fit_iv,
                                              ANN_BATCH_COA,shuffle=False),
                  epochs=8, class_weight=cw,
                  callbacks=[EarlyStopping(monitor="val_loss",patience=1,
                             restore_best_weights=True,verbose=0)],
                  verbose=0)
        prob = model.predict(val_ds_ann,verbose=0).flatten()
        f1   = float(f1_score(y_val,(prob>0.5).astype(int)))   # <- y_val, not y_test
        print(f"  eval{eval_cnt[0]:03d} h1={h1} h2={h2} lr={lr:.6f} -> F1={f1:.4f}")
        del model; gc.collect(); tf.keras.backend.clear_session()
        return f1

def coa(fitness_fn, bounds, n_coyotes=8, n_iters=8,
        p_leave=0.1, rng_seed=SEED):
    rng = np.random.RandomState(rng_seed)
    pop  = [rand_coyote(bounds,rng) for _ in range(n_coyotes)]
    fits = [fitness_fn(p) for p in pop]
    bi   = int(np.argmax(fits)); best_pos=pop[bi].copy(); best_fit=fits[bi]
    hist = [round(best_fit,4)]
    print(f"Initial best: {best_fit:.4f}")
    for it in range(n_iters):
        print(f"\nIteration {it+1}/{n_iters} | best={best_fit:.4f}")
        alpha  = pop[int(np.argmax(fits))].copy()
        tend   = np.median(pop,axis=0)
        for i in range(n_coyotes):
            if i==int(np.argmax(fits)): continue
            r1,r2  = rng.rand(),rng.rand()
            new_p  = clip_pos(pop[i]+r1*(alpha-pop[i])+r2*(tend-pop[i]),bounds)
            new_f  = fitness_fn(new_p)
            if new_f > fits[i]: pop[i]=new_p; fits[i]=new_f
        if rng.rand()<p_leave:
            wi=int(np.argmin(fits)); cp=rand_coyote(bounds,rng); cf=fitness_fn(cp)
            if cf>fits[wi]: pop[wi]=cp; fits[wi]=cf
        bi=int(np.argmax(fits))
        if fits[bi]>best_fit: best_fit=fits[bi]; best_pos=pop[bi].copy()
        hist.append(round(best_fit,4))
    return best_pos, best_fit, hist

print("Running COA-ANN (8 coyotes x 8 iters, ~72 evals, fitness on VAL set)...")
t0=time.time()
ann_best_pos, ann_best_fit, ann_history = coa(ann_fitness, ANN_BOUNDS,
                                               n_coyotes=8, n_iters=8)
h1_best, h2_best, lr_ann_best = decode_ann(ann_best_pos)
print(f"\nCOA-ANN DONE in {(time.time()-t0)/60:.1f} min: "
      f"H1={h1_best}  H2={h2_best}  LR={lr_ann_best:.8f}  "
      f"val_F1={ann_best_fit:.4f}  evals={eval_cnt[0]}")

coa_ann_result = {"best_h1":h1_best,"best_h2":h2_best,"best_lr":lr_ann_best,
                  "best_val_f1":round(ann_best_fit,4),
                  "n_coyotes":8,"n_iters":8,"n_evals":eval_cnt[0],
                  "convergence":ann_history}
json.dump(coa_ann_result,open(f"{RESULTS_DIR}/coa_ann_result.json","w"),indent=2)


## 7. COA — CNN+BiLSTM Hyperparameter Search\n\nSame leakage rule applies: fitness uses `y_val` only.

In [ ]:
CNN_BOUNDS = np.array([[64,256],[32,128],[-4,-2]], dtype=float)

def decode_cnn(pos):
    return (int(round(np.clip(pos[0],64,256))),
            int(round(np.clip(pos[1],32,128))),
            float(10**np.clip(pos[2],-4,-2)))

# 15k stratified fitness subset for CNN speed (small dataset -> small subset)
CNN_FIT_SUBSET = min(15000, int(len(y_train)*0.8))
sub_idx_cnn,_ = train_test_split(np.arange(len(y_train)),
                                  train_size=CNN_FIT_SUBSET,
                                  stratify=y_train,random_state=SEED)
X_seq_sub = X_seq_train[sub_idx_cnn]; y_sub = y_train[sub_idx_cnn]

val_ds_cnn = make_seq_ds(X_seq_val, np.zeros(len(y_val)),
                          CNN_BATCH_COA*4, shuffle=False)
n_iv_cnn   = int(len(y_sub)*0.1)
X_csub_tr  = X_seq_sub[:-n_iv_cnn]; y_csub_tr = y_sub[:-n_iv_cnn]
X_csub_iv  = X_seq_sub[-n_iv_cnn:]; y_csub_iv = y_sub[-n_iv_cnn:]
eval_cnt_cnn = [0]

def cnn_fitness(pos):
    """Trains on the 15k subset, evaluates F1 on X_seq_val/y_val.
    y_test is NEVER referenced in this function. Runs on COA_DEVICE (single GPU)
    so it never restricts global GPU visibility for later MirroredStrategy use."""
    with tf.device(COA_DEVICE):
        eval_cnt_cnn[0] += 1
        filters,lstm_u,lr = decode_cnn(pos)
        set_seeds_tf(SEED + eval_cnt_cnn[0] + 200)
        model = tf.keras.Sequential([
            tf.keras.layers.Embedding(vocab_size,EMBED_DIM,
                                       weights=[emb_mat],trainable=False),
            tf.keras.layers.Conv1D(filters,3,activation="relu"),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_u)),
            tf.keras.layers.Dense(64,activation="relu"),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32"),
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss="binary_crossentropy",metrics=["accuracy"])
        model.fit(make_seq_ds(X_csub_tr,y_csub_tr,CNN_BATCH_COA),
                  validation_data=make_seq_ds(X_csub_iv,y_csub_iv,
                                               CNN_BATCH_COA,shuffle=False),
                  epochs=5, class_weight=cw,
                  callbacks=[EarlyStopping(monitor="val_loss",patience=1,
                             restore_best_weights=True,verbose=0)],
                  verbose=0)
        prob = model.predict(val_ds_cnn,verbose=0).flatten()
        f1   = float(f1_score(y_val,(prob>0.5).astype(int)))   # <- y_val, not y_test
        print(f"  eval{eval_cnt_cnn[0]:03d} f={filters} lstm={lstm_u} lr={lr:.6f} -> F1={f1:.4f}")
        del model; gc.collect(); tf.keras.backend.clear_session()
        return f1

print(f"Running COA-CNN (6 coyotes x 4 iters, {CNN_FIT_SUBSET:,} fitness subset)...")
t0=time.time()
cnn_best_pos, cnn_best_fit, cnn_history = coa(cnn_fitness, CNN_BOUNDS,
                                               n_coyotes=6, n_iters=4,
                                               rng_seed=SEED+100)
f_best, lstm_best, lr_cnn_best = decode_cnn(cnn_best_pos)
print(f"\nCOA-CNN DONE in {(time.time()-t0)/60:.1f} min: "
      f"F={f_best}  LSTM={lstm_best}  LR={lr_cnn_best:.8f}  "
      f"val_F1={cnn_best_fit:.4f}  evals={eval_cnt_cnn[0]}")

coa_cnn_result = {"best_filters":f_best,"best_lstm_units":lstm_best,
                  "best_lr":lr_cnn_best,"best_val_f1":round(cnn_best_fit,4),
                  "n_coyotes":6,"n_iters":4,"n_evals":eval_cnt_cnn[0],
                  "fitness_subset":CNN_FIT_SUBSET,"convergence":cnn_history}
json.dump(coa_cnn_result,open(f"{RESULTS_DIR}/coa_cnn_result.json","w"),indent=2)


## 8. Final Training — TF-IDF, ANN, CNN+BiLSTM (MirroredStrategy)

In [ ]:
tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy  = tf.distribute.MirroredStrategy()
N_GPUS    = strategy.num_replicas_in_sync
ANN_BATCH = 512 * N_GPUS
CNN_BATCH = 64  * N_GPUS
print(f"MirroredStrategy: {N_GPUS} GPU(s)  ANN_BATCH={ANN_BATCH}  CNN_BATCH={CNN_BATCH}")

X_emb_tv = np.vstack([X_emb_train, X_emb_val])
X_seq_tv  = np.vstack([X_seq_train, X_seq_val])
y_tv      = np.concatenate([y_train, y_val])
X_txt_tv  = X_str_train + X_str_val
cw_tv = dict(zip(
    [int(k) for k in np.unique(y_tv)],
    [float(v) for v in compute_class_weight("balanced",
                        classes=np.unique(y_tv),y=y_tv)]))
n_es = max(int(len(y_tv)*0.1), 8)

set_seeds_tf()
all_results = []; all_probs = {}

def train_ann_final(name, h1, h2, lr, seed=SEED):
    print(f"\nTraining {name}: H1={h1} H2={h2} LR={lr:.8f}  batch={ANN_BATCH}")
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1,activation="relu",input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2,activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy",metrics=["accuracy"])
    ckpt = f"{MODELS_DIR}/{name.replace(' ','_').replace('+','').lower()}.keras"
    m.fit(make_tf_ds(X_emb_tv[:-n_es],y_tv[:-n_es],ANN_BATCH),
          validation_data=make_tf_ds(X_emb_tv[-n_es:],
                                      y_tv[-n_es:],ANN_BATCH,shuffle=False),
          epochs=100, class_weight=cw_tv,
          callbacks=[make_es(10,verbose=0),
                     ModelCheckpoint(ckpt,monitor="val_loss",
                                     save_best_only=True,verbose=0)],
          verbose=0)
    m = tf.keras.models.load_model(ckpt)
    prob = m.predict(make_tf_ds(X_emb_test,np.zeros(len(y_test)),
                                 ANN_BATCH,shuffle=False),verbose=0).flatten()
    pred = (prob>0.5).astype(int)
    r    = evaluate(name,y_test,pred,prob)
    bt   = best_threshold(y_test,prob)
    r["optimal_threshold"] = round(bt,4)
    print(f"  F1={r['F1-score']}  MCC={r['MCC']}  AUC={r['AUC-ROC']}")
    return r, prob

def train_cnn_final(name, filters, lstm_u, lr, seed=SEED):
    print(f"\nTraining {name}: F={filters} LSTM={lstm_u} LR={lr:.8f}  batch={CNN_BATCH}")
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Embedding(vocab_size,EMBED_DIM,
                                       weights=[emb_mat],trainable=False),
            tf.keras.layers.Conv1D(filters,3,activation="relu"),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_u)),
            tf.keras.layers.Dense(64,activation="relu"),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy",metrics=["accuracy"])
    ckpt = f"{MODELS_DIR}/{name.replace(' ','_').replace('+','').lower()}.keras"
    m.fit(make_seq_ds(X_seq_tv[:-n_es],y_tv[:-n_es],CNN_BATCH),
          validation_data=make_seq_ds(X_seq_tv[-n_es:],
                                       y_tv[-n_es:],CNN_BATCH,shuffle=False),
          epochs=30, class_weight=cw_tv,
          callbacks=[make_es(5,verbose=0),
                     ModelCheckpoint(ckpt,monitor="val_loss",
                                     save_best_only=True,verbose=0)],
          verbose=0)
    m = tf.keras.models.load_model(ckpt)
    prob = m.predict(make_seq_ds(X_seq_test,np.zeros(len(y_test)),
                                  CNN_BATCH,shuffle=False),verbose=0).flatten()
    pred = (prob>0.5).astype(int)
    r    = evaluate(name,y_test,pred,prob)
    bt   = best_threshold(y_test,prob)
    r["optimal_threshold"] = round(bt,4)
    print(f"  F1={r['F1-score']}  MCC={r['MCC']}  AUC={r['AUC-ROC']}")
    return r, prob

print("Training TF-IDF + LR...")
lr_pipe = Pipeline([
    ("tfidf",TfidfVectorizer(max_features=20000,ngram_range=(1,2),sublinear_tf=True)),
    ("clf",  LogisticRegression(max_iter=1000,solver="lbfgs",random_state=SEED))])
lr_pipe.fit(X_txt_tv,y_tv)
lr_pred  = lr_pipe.predict(X_str_test)
lr_prob  = lr_pipe.predict_proba(X_str_test)[:,1]
r = evaluate("TF-IDF + LR",y_test,lr_pred,lr_prob)
all_results.append(r); all_probs["TF-IDF + LR"]=lr_prob
print(f"  F1={r['F1-score']}  MCC={r['MCC']}")

print("Training TF-IDF + SVM...")
svm_pipe = Pipeline([
    ("tfidf",TfidfVectorizer(max_features=20000,ngram_range=(1,2),sublinear_tf=True)),
    ("clf",  LinearSVC(max_iter=3000,random_state=SEED,dual=False))])
svm_pipe.fit(X_txt_tv,y_tv)
svm_pred   = svm_pipe.predict(X_str_test)
svm_scores = svm_pipe.decision_function(X_str_test)
r = evaluate("TF-IDF + SVM",y_test,svm_pred,svm_scores)
all_results.append(r); all_probs["TF-IDF + SVM"]=svm_scores
print(f"  F1={r['F1-score']}  MCC={r['MCC']}")

r,prob = train_ann_final("Default ANN",128,64,1e-3)
all_results.append(r); all_probs["Default ANN"]=prob

r,prob = train_ann_final("COA-ANN",h1_best,h2_best,lr_ann_best)
all_results.append(r); all_probs["COA-ANN"]=prob

r,prob = train_cnn_final("Default CNN+BiLSTM",128,64,1e-3)
all_results.append(r); all_probs["Default CNN+BiLSTM"]=prob

r,prob = train_cnn_final("COA-CNN+BiLSTM",f_best,lstm_best,lr_cnn_best)
all_results.append(r); all_probs["COA-CNN+BiLSTM"]=prob

print("\nDeep learning + TF-IDF models trained.")


## 9. AdaBoost on FastText Embeddings\n\nUses the same `X_emb_tv` / `X_emb_test` features as the ANN models — directly comparable baseline. Sequential by nature (single core), so kept lightweight given the small 36k-sample training pool.

In [ ]:
print(f"Training AdaBoost on {len(y_tv):,} samples (full subset — small enough to use directly)...")

print("\n[1/2] AdaBoost (depth-1 stumps, n_estimators=150)...")
t0=time.time()
set_seeds()
ada_stumps = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=SEED),
    n_estimators=150, learning_rate=1.0, random_state=SEED)
ada_stumps.fit(X_emb_tv, y_tv)
prob_stumps = ada_stumps.predict_proba(X_emb_test)[:,1]
pred_stumps = (prob_stumps > 0.5).astype(int)
r = evaluate("AdaBoost (depth-1, n=150)", y_test, pred_stumps, prob_stumps)
all_results.append(r); all_probs["AdaBoost (depth-1, n=150)"] = prob_stumps
print(f"  Done in {time.time()-t0:.1f}s | F1={r['F1-score']}  MCC={r['MCC']}  AUC={r['AUC-ROC']}")

print("\n[2/2] AdaBoost (depth-3 trees, n_estimators=75)...")
t0=time.time()
set_seeds()
ada_depth3 = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3, random_state=SEED),
    n_estimators=75, learning_rate=1.0, random_state=SEED)
ada_depth3.fit(X_emb_tv, y_tv)
prob_depth3 = ada_depth3.predict_proba(X_emb_test)[:,1]
pred_depth3 = (prob_depth3 > 0.5).astype(int)
r = evaluate("AdaBoost (depth-3, n=75)", y_test, pred_depth3, prob_depth3)
all_results.append(r); all_probs["AdaBoost (depth-3, n=75)"] = prob_depth3
print(f"  Done in {time.time()-t0:.1f}s | F1={r['F1-score']}  MCC={r['MCC']}  AUC={r['AUC-ROC']}")

with open(f"{MODELS_DIR}/adaboost_stumps.pkl","wb") as f: pickle.dump(ada_stumps,f)
with open(f"{MODELS_DIR}/adaboost_depth3.pkl","wb") as f: pickle.dump(ada_depth3,f)
np.save(f"{RESULTS_DIR}/adaboost_stumps_prob.npy", prob_stumps)
np.save(f"{RESULTS_DIR}/adaboost_depth3_prob.npy", prob_depth3)

print("\nAdaBoost training complete.")


## 10. Save Full Comparison

In [ ]:
results_df = (pd.DataFrame(all_results)
                .sort_values("F1-score",ascending=False)
                .reset_index(drop=True))
results_df.index += 1
results_df.to_csv(f"{RESULTS_DIR}/final_comparison.csv")
for name,prob in all_probs.items():
    fname = name.replace(' ','_').replace('+','').replace('(','').replace(')','').replace(',','').replace('=','').lower()
    np.save(f"{RESULTS_DIR}/{fname}_prob.npy",prob)
json.dump({r["Model"]:{k:v for k,v in r.items() if k!="Model"} for r in all_results},
          open(f"{RESULTS_DIR}/final_metrics.json","w"),indent=2)

print("=== FULL RESULTS (sorted by F1) ===")
print(results_df[["Model","F1-score","Macro-F1","MCC","AUC-ROC"]].to_string())


## 11. Ablation Study

In [ ]:
# 2026-10-12: the "Full" rows are retrained below with the SAME schedule as the ablated variants
# (previously copied from the ≤100/30-epoch final-training run).
ablation = []

def abl_ann(name, h1, h2, lr, zero_emb=False, shallow=False, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        if shallow:
            m = tf.keras.Sequential([
                tf.keras.layers.Dense(h1,activation="relu",input_shape=(EMBED_DIM,)),
                tf.keras.layers.Dropout(0.3),
                tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32")])
        else:
            m = tf.keras.Sequential([
                tf.keras.layers.Dense(h1,activation="relu",input_shape=(EMBED_DIM,)),
                tf.keras.layers.Dropout(0.3),
                tf.keras.layers.Dense(h2,activation="relu"),
                tf.keras.layers.Dropout(0.3),
                tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32")])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy",metrics=["accuracy"])
    X_tr = np.zeros_like(X_emb_tv) if zero_emb else X_emb_tv
    X_te = np.zeros_like(X_emb_test) if zero_emb else X_emb_test
    m.fit(make_tf_ds(X_tr[:-n_es],y_tv[:-n_es],ANN_BATCH),
          validation_data=make_tf_ds(X_tr[-n_es:],y_tv[-n_es:],ANN_BATCH,shuffle=False),
          epochs=50,class_weight=cw_tv,callbacks=[make_es(5)],verbose=0)
    prob=m.predict(make_tf_ds(X_te,np.zeros(len(y_test)),ANN_BATCH,shuffle=False),verbose=0).flatten()
    r=evaluate(name,y_test,(prob>0.5).astype(int),prob)
    print(f"  {name}: F1={r['F1-score']}  MCC={r['MCC']}"); return r

def abl_cnn(name, filters, lstm_u, lr, pretrained=True,
            use_bilstm=True, use_dropout=True, seed=SEED):
    """NOTE on pretrained=False: a large randomly-initialized trainable
    Embedding layer combined with mixed_float16 + MirroredStrategy can trigger
    a GPU-side InvalidArgumentError inside the loss-scale optimizer's
    check_finite/TensorScatterUpdate step (a known TF/Keras GPU interaction,
    unrelated to input data correctness). We sidestep it by running ONLY the
    pretrained=False variant outside mixed precision and outside the
    multi-GPU strategy scope -- single GPU, float32. This does not change
    what is being measured (embedding ablation), only the numerical execution
    path used to compute it."""
    set_seeds_tf(seed)

    def build_layers():
        emb = (tf.keras.layers.Embedding(vocab_size,EMBED_DIM,
                                          weights=[emb_mat],trainable=False)
               if pretrained else
               tf.keras.layers.Embedding(vocab_size,EMBED_DIM,trainable=True))
        layers = [emb, tf.keras.layers.Conv1D(filters,3,activation="relu"),
                  tf.keras.layers.MaxPooling1D(2)]
        if use_bilstm:
            layers.append(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_u)))
        else:
            layers.append(tf.keras.layers.GlobalMaxPooling1D())
        layers.append(tf.keras.layers.Dense(64,activation="relu"))
        if use_dropout:
            layers.append(tf.keras.layers.Dropout(0.5))
        layers.append(tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32"))
        return layers

    if pretrained:
        # Normal path: frozen pretrained embedding, full mixed-precision + multi-GPU
        with strategy.scope():
            m = tf.keras.Sequential(build_layers())
            m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                      loss="binary_crossentropy",metrics=["accuracy"])
        m.fit(make_seq_ds(X_seq_tv[:-n_es],y_tv[:-n_es],CNN_BATCH),
              validation_data=make_seq_ds(X_seq_tv[-n_es:],y_tv[-n_es:],CNN_BATCH,shuffle=False),
              epochs=20,class_weight=cw_tv,callbacks=[make_es(3)],verbose=0)
        prob=m.predict(make_seq_ds(X_seq_test,np.zeros(len(y_test)),CNN_BATCH,shuffle=False),verbose=0).flatten()
    else:
        # w/o FastText path: random trainable embedding -- run single-GPU, float32,
        # no MirroredStrategy scope, to avoid the GPU scatter-update crash.
        prev_policy = tf.keras.mixed_precision.global_policy()
        tf.keras.mixed_precision.set_global_policy("float32")
        try:
            with tf.device(COA_DEVICE):
                layers = build_layers()
                m = tf.keras.Sequential(layers)
                m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                          loss="binary_crossentropy",metrics=["accuracy"])
                m.fit(make_seq_ds(X_seq_tv[:-n_es],y_tv[:-n_es],CNN_BATCH_COA),
                      validation_data=make_seq_ds(X_seq_tv[-n_es:],y_tv[-n_es:],CNN_BATCH_COA,shuffle=False),
                      epochs=20,class_weight=cw_tv,callbacks=[make_es(3)],verbose=0)
                prob=m.predict(make_seq_ds(X_seq_test,np.zeros(len(y_test)),CNN_BATCH_COA,shuffle=False),verbose=0).flatten()
        finally:
            tf.keras.mixed_precision.set_global_policy(prev_policy)

    r=evaluate(name,y_test,(prob>0.5).astype(int),prob)
    print(f"  {name}: F1={r['F1-score']}  MCC={r['MCC']}"); return r

print("ANN ablation...")
ablation.append(abl_ann("Full COA-ANN",h1_best,h2_best,lr_ann_best))
ablation.append(abl_ann("w/o COA (Default ANN)",128,64,1e-3))
ablation.append(abl_ann("Shallow (1 layer)",h1_best,h2_best,lr_ann_best,shallow=True))
ablation.append(abl_ann("w/o FastText (zeros)",h1_best,h2_best,lr_ann_best,zero_emb=True))

print("\nCNN ablation...")
ablation.append(abl_cnn("Full COA-CNN+BiLSTM",f_best,lstm_best,lr_cnn_best))
ablation.append(abl_cnn("w/o COA (Default CNN)",128,64,1e-3))
ablation.append(abl_cnn("w/o FastText (random)",f_best,lstm_best,lr_cnn_best,pretrained=False))
ablation.append(abl_cnn("w/o BiLSTM (CNN only)",f_best,lstm_best,lr_cnn_best,use_bilstm=False))
ablation.append(abl_cnn("w/o Dropout",f_best,lstm_best,lr_cnn_best,use_dropout=False))

pd.DataFrame(ablation).to_csv(f"{RESULTS_DIR}/ablation_results.csv",index=False)
print("\nAblation saved.")

## 12. Wilcoxon Statistical Test (5 seeds, ANN + CNN)

In [ ]:
STAT_SEEDS = [42, 7, 13, 99, 2024, 11, 23, 57, 101, 314]   # 2026-10-12: 5 → 10 seeds
f1_coa_ann=[]; f1_def_ann=[]; f1_coa_cnn=[]; f1_def_cnn=[]

for seed in STAT_SEEDS:
    print(f"\nSeed {seed}:")
    set_seeds_tf(seed)
    with strategy.scope():
        m=tf.keras.Sequential([
            tf.keras.layers.Dense(h1_best,activation="relu",input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2_best,activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32")])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr_ann_best),
                  loss="binary_crossentropy",metrics=["accuracy"])
    m.fit(make_tf_ds(X_emb_tv[:-n_es],y_tv[:-n_es],ANN_BATCH),
          validation_data=make_tf_ds(X_emb_tv[-n_es:],y_tv[-n_es:],ANN_BATCH,shuffle=False),
          epochs=50,class_weight=cw_tv,callbacks=[make_es(5)],verbose=0)
    prob=m.predict(make_tf_ds(X_emb_test,np.zeros(len(y_test)),ANN_BATCH,shuffle=False),verbose=0).flatten()
    f=float(f1_score(y_test,(prob>0.5).astype(int))); f1_coa_ann.append(f)
    print(f"  COA-ANN={f:.4f}",end="  ")

    set_seeds_tf(seed+500)
    with strategy.scope():
        m=tf.keras.Sequential([
            tf.keras.layers.Dense(128,activation="relu",input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64,activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32")])
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",metrics=["accuracy"])
    m.fit(make_tf_ds(X_emb_tv[:-n_es],y_tv[:-n_es],ANN_BATCH),
          validation_data=make_tf_ds(X_emb_tv[-n_es:],y_tv[-n_es:],ANN_BATCH,shuffle=False),
          epochs=50,class_weight=cw_tv,callbacks=[make_es(5)],verbose=0)
    prob=m.predict(make_tf_ds(X_emb_test,np.zeros(len(y_test)),ANN_BATCH,shuffle=False),verbose=0).flatten()
    f=float(f1_score(y_test,(prob>0.5).astype(int))); f1_def_ann.append(f)
    print(f"Def-ANN={f:.4f}")

    set_seeds_tf(seed+1000)
    with strategy.scope():
        m=tf.keras.Sequential([
            tf.keras.layers.Embedding(vocab_size,EMBED_DIM,weights=[emb_mat],trainable=False),
            tf.keras.layers.Conv1D(f_best,3,activation="relu"),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_best)),
            tf.keras.layers.Dense(64,activation="relu"),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32")])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr_cnn_best),
                  loss="binary_crossentropy",metrics=["accuracy"])
    m.fit(make_seq_ds(X_seq_tv[:-n_es],y_tv[:-n_es],CNN_BATCH),
          validation_data=make_seq_ds(X_seq_tv[-n_es:],y_tv[-n_es:],CNN_BATCH,shuffle=False),
          epochs=20,class_weight=cw_tv,callbacks=[make_es(3)],verbose=0)
    prob=m.predict(make_seq_ds(X_seq_test,np.zeros(len(y_test)),CNN_BATCH,shuffle=False),verbose=0).flatten()
    f=float(f1_score(y_test,(prob>0.5).astype(int))); f1_coa_cnn.append(f)
    print(f"  COA-CNN={f:.4f}",end="  ")

    set_seeds_tf(seed+1500)
    with strategy.scope():
        m=tf.keras.Sequential([
            tf.keras.layers.Embedding(vocab_size,EMBED_DIM,weights=[emb_mat],trainable=False),
            tf.keras.layers.Conv1D(128,3,activation="relu"),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
            tf.keras.layers.Dense(64,activation="relu"),
            tf.keras.layers.Dropout(0.5),
            tf.keras.layers.Dense(1,activation="sigmoid",dtype="float32")])
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",metrics=["accuracy"])
    m.fit(make_seq_ds(X_seq_tv[:-n_es],y_tv[:-n_es],CNN_BATCH),
          validation_data=make_seq_ds(X_seq_tv[-n_es:],y_tv[-n_es:],CNN_BATCH,shuffle=False),
          epochs=20,class_weight=cw_tv,callbacks=[make_es(3)],verbose=0)
    prob=m.predict(make_seq_ds(X_seq_test,np.zeros(len(y_test)),CNN_BATCH,shuffle=False),verbose=0).flatten()
    f=float(f1_score(y_test,(prob>0.5).astype(int))); f1_def_cnn.append(f)
    print(f"Def-CNN={f:.4f}")

def _safe_wilcoxon(a, b):   # 2026-10-12: SciPy raises if every paired difference is exactly zero
    d = np.array(a) - np.array(b)
    return wilcoxon(a, b) if np.any(d != 0) else (float("nan"), 1.0)
stat_a,p_a = _safe_wilcoxon(f1_coa_ann,f1_def_ann)
stat_c,p_c = _safe_wilcoxon(f1_coa_cnn,f1_def_cnn)

wilcoxon_result = {
    "ann":{"coa_f1":f1_coa_ann,"def_f1":f1_def_ann,
           "coa_mean":round(float(np.mean(f1_coa_ann)),4),
           "def_mean":round(float(np.mean(f1_def_ann)),4),
           "coa_std" :round(float(np.std(f1_coa_ann)),4),
           "def_std" :round(float(np.std(f1_def_ann)),4),
           "n_seeds":len(STAT_SEEDS),"p_value" :round(float(p_a),6),"significant":bool(p_a<0.05)},
    "cnn":{"coa_f1":f1_coa_cnn,"def_f1":f1_def_cnn,
           "coa_mean":round(float(np.mean(f1_coa_cnn)),4),
           "def_mean":round(float(np.mean(f1_def_cnn)),4),
           "coa_std" :round(float(np.std(f1_coa_cnn)),4),
           "def_std" :round(float(np.std(f1_def_cnn)),4),
           "n_seeds":len(STAT_SEEDS),"p_value" :round(float(p_c),6),"significant":bool(p_c<0.05)},
}
json.dump(wilcoxon_result,open(f"{RESULTS_DIR}/wilcoxon_result.json","w"),indent=2)
print(f"\nANN: COA={np.mean(f1_coa_ann):.4f}±{np.std(f1_coa_ann):.4f}  "
      f"Def={np.mean(f1_def_ann):.4f}±{np.std(f1_def_ann):.4f}  p={p_a:.4f}")
print(f"CNN: COA={np.mean(f1_coa_cnn):.4f}±{np.std(f1_coa_cnn):.4f}  "
      f"Def={np.mean(f1_def_cnn):.4f}±{np.std(f1_def_cnn):.4f}  p={p_c:.4f}")


## 13. Figures

In [ ]:
MODEL_NAMES = ["TF-IDF + LR","TF-IDF + SVM","Default ANN","COA-ANN",
               "Default CNN+BiLSTM","COA-CNN+BiLSTM",
               "AdaBoost (depth-1, n=150)","AdaBoost (depth-3, n=75)"]
COLORS = ["#e74c3c","#e67e22","#3498db","#9b59b6","#1abc9c","#2c3e50","#f1c40f","#7f8c8d"]
STYLES = ["-","--","-.",":","--","-","-.",":"]

fig, ax = plt.subplots(figsize=(13,5))
x = np.arange(len(results_df)); w = 0.18
for i,(m,c) in enumerate(zip(["F1-score","Macro-F1","MCC","AUC-ROC"],
                               ["#e74c3c","#2ecc71","#f39c12","#3498db"])):
    ax.bar(x+i*w, results_df[m], w, label=m, color=c, alpha=0.85)
ax.set_xticks(x+1.5*w)
ax.set_xticklabels(results_df["Model"],rotation=25,ha="right",fontsize=8)
ax.set_ylim(0.3,1.05); ax.set_ylabel("Score")
ax.set_title(f"Amazon Fine Food Reviews ({N_NEG+N_POS:,} samples) — All Models incl. AdaBoost")
ax.legend(loc="lower right"); ax.grid(axis="y",alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_comparison.pdf",dpi=300,bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1,2,figsize=(13,4))
for ax,hist,title,col,cfg_r in [
    (axes[0],ann_history,"COA-ANN","#e74c3c",coa_ann_result),
    (axes[1],cnn_history,"COA-CNN+BiLSTM","#3498db",coa_cnn_result),
]:
    ax.plot(range(len(hist)),hist,"o-",color=col,lw=2,ms=7)
    ax.set_title(f"{title} Convergence\nBest val F1={cfg_r['best_val_f1']}")
    ax.set_xlabel("Iteration (0=init)"); ax.set_ylabel("Best Val F1")
    ax.grid(alpha=0.3)
    for i,v in enumerate(hist):
        ax.annotate(f"{v:.4f}",(i,v),textcoords="offset points",
                    xytext=(0,7),ha="center",fontsize=7)
plt.suptitle("COA Convergence — Amazon Fine Food Reviews",fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_coa_convergence.pdf",dpi=300,bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1,2,figsize=(13,5))
for name,col,ls in zip(MODEL_NAMES,COLORS,STYLES):
    prob = all_probs[name]
    fpr,tpr,_ = roc_curve(y_test,prob)
    auc = roc_auc_score(y_test,prob)
    axes[0].plot(fpr,tpr,ls,color=col,lw=2,label=f"{name} ({auc:.4f})")
    prec,rec,_ = precision_recall_curve(y_test,prob)
    ap = average_precision_score(y_test,prob)
    axes[1].plot(rec,prec,ls,color=col,lw=2,label=f"{name} ({ap:.4f})")
axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=0.4)
axes[0].set_title("ROC Curves"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].legend(fontsize=6,loc="lower right"); axes[0].grid(alpha=0.3)
axes[1].axhline(y_test.sum()/len(y_test),color="grey",ls=":",lw=1.5)
axes[1].set_title("PR Curves"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=6,loc="upper right"); axes[1].grid(alpha=0.3)
plt.suptitle("Amazon Fine Food Reviews — ROC & PR Curves",fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_roc_pr.pdf",dpi=300,bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax,name in zip(axes,["COA-ANN","COA-CNN+BiLSTM"]):
    pred = (all_probs[name]>0.5).astype(int)
    cm = confusion_matrix(y_test,pred)
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax,
                xticklabels=["Neg","Pos"],yticklabels=["Neg","Pos"],linewidths=0.5)
    ax.set_title(f"{name}\nF1={f1_score(y_test,pred):.4f}  "
                 f"MCC={matthews_corrcoef(y_test,pred):.4f}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices — COA Models",fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_confusion_matrices.pdf",dpi=300,bbox_inches="tight")
plt.show()

abl_df = pd.DataFrame(ablation)
fig, axes = plt.subplots(1,2,figsize=(15,5))
for ax,key,title in [
    (axes[0],["Full COA-ANN","w/o COA (Default ANN)","Shallow (1 layer)","w/o FastText (zeros)"],"ANN"),
    (axes[1],["Full COA-CNN+BiLSTM","w/o COA (Default CNN)","w/o FastText (random)","w/o BiLSTM (CNN only)","w/o Dropout"],"CNN"),
]:
    df_ = abl_df[abl_df["Model"].isin(key)].set_index("Model").reindex(key)
    x_  = np.arange(len(df_)); w_ = 0.2
    for i,(m,c) in enumerate(zip(["F1-score","Macro-F1","MCC","AUC-ROC"],
                                   ["#e74c3c","#2ecc71","#f39c12","#3498db"])):
        ax.bar(x_+i*w_, df_[m], w_, label=m, color=c, alpha=0.85)
    ax.set_xticks(x_+1.5*w_)
    ax.set_xticklabels(df_.index,rotation=15,ha="right",fontsize=8)
    ax.set_ylim(0.3,1.05); ax.set_title(f"{title} Ablation"); ax.legend(fontsize=8)
    ax.set_ylabel("Score"); ax.grid(axis="y",alpha=0.3)
plt.suptitle("Ablation Study — Amazon Fine Food Reviews",fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_ablation.pdf",dpi=300,bbox_inches="tight")
plt.show()

labels_w = ["COA-ANN\nvs Default","COA-CNN\nvs Default"]
pvals    = [wilcoxon_result["ann"]["p_value"],wilcoxon_result["cnn"]["p_value"]]
fig, ax  = plt.subplots(figsize=(6,4))
ax.bar(labels_w, pvals,
       color=["#2ecc71" if p<0.05 else "#e74c3c" for p in pvals], alpha=0.85)
ax.axhline(0.05,color="black",ls="--",lw=2,label="α=0.05")
ax.set_ylabel("p-value"); ax.set_title(f"Wilcoxon p-values ({len(STAT_SEEDS)} seeds)")
ax.legend(); ax.grid(axis="y",alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_wilcoxon.pdf",dpi=300,bbox_inches="tight")
plt.show()

print("All figures saved.")


## 14. Final Summary

In [ ]:
cfg = json.load(open(f"{SHARED_DIR}/config.json"))
print("="*60)
print("FINAL SUMMARY — Amazon Fine Food Reviews (small subset)")
print("="*60)
print(f"Subset: {N_NEG+N_POS:,}  (neg={N_NEG:,} pos={N_POS:,})")
print(f"Split: train={cfg['n_train']:,}  val={cfg['n_val']:,}  test={cfg['n_test']:,}")
print(f"MAX_LEN={cfg['max_len']}  vocab={cfg['vocab_size']:,}  emb_cov={cfg['emb_coverage_pct']}%")
print(f"\nCOA-ANN : H1={h1_best}  H2={h2_best}  LR={lr_ann_best:.8f}  val_F1={coa_ann_result['best_val_f1']}")
print(f"COA-CNN : F={f_best}  LSTM={lstm_best}  LR={lr_cnn_best:.8f}  val_F1={coa_cnn_result['best_val_f1']}")

print(f"\nTest Set Results (all models, including AdaBoost):")
for _,row in results_df.iterrows():
    print(f"  {row['Model']:28s}: F1={row['F1-score']}  MCC={row['MCC']}  AUC={row['AUC-ROC']}")

best_overall = results_df.iloc[0]
print(f"\nBest overall: {best_overall['Model']}  (F1={best_overall['F1-score']})")

ada_best = max([r for r in all_results if "AdaBoost" in r["Model"]], key=lambda r: r["F1-score"])
coa_ann_test = next(r for r in all_results if r["Model"]=="COA-ANN")
print(f"\nAdaBoost best variant: {ada_best['Model']} (F1={ada_best['F1-score']})")
print(f"COA-ANN F1: {coa_ann_test['F1-score']}")
diff = ada_best["F1-score"] - coa_ann_test["F1-score"]
print(f"AdaBoost {'beats' if diff>0 else 'underperforms'} COA-ANN by {abs(diff):.4f} F1")

print(f"\nWilcoxon ({len(STAT_SEEDS)} seeds, ANN+CNN only):")
for mt in ["ann","cnn"]:
    s = wilcoxon_result[mt]
    print(f"  COA-{mt.upper()}: COA={s['coa_mean']}±{s['coa_std']}  Def={s['def_mean']}±{s['def_std']}  p={s['p_value']}")

print(f"\nFiles saved to {RESULTS_DIR}/")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  ✓ {f}")
